# Dimensionality Reduction — PCA + UMAP

Validate ML pipeline visually:
- PCA: are clusters well-separated? Which sensors explain variance?
- UMAP: do failures form distinct islands?
- Anomaly overlay: are anomalies peripheral?

**Notebook only** — not deployed to production.

In [ ]:
import os, sys, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns, joblib, warnings
warnings.filterwarnings('ignore')
ROOT = os.path.abspath('..')
if ROOT not in sys.path: sys.path.insert(0, ROOT)
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
try:
    import umap; HAS_UMAP = True; print('UMAP available')
except ImportError:
    HAS_UMAP = False; print('UMAP not found. pip install umap-learn')
sns.set_theme(style='whitegrid'); plt.rcParams['figure.figsize'] = (14,6); print('Setup OK')

In [ ]:
CSV_PATH = os.path.join(ROOT, 'ai4i2020.csv')
df = pd.read_csv(CSV_PATH)
SENSOR_COLS = ['Air temperature [K]','Process temperature [K]',
               'Rotational speed [rpm]','Torque [Nm]','Tool wear [min]']
X_raw = df[SENSOR_COLS].values
y_failure = df['Machine failure'].values
MODELS_DIR = os.path.join(ROOT, 'app', 'backend', 'modules', 'ml', 'models')
pipeline_path = os.path.join(MODELS_DIR, 'feature_pipeline_v3.pkl')
if os.path.exists(pipeline_path):
    pipeline = joblib.load(pipeline_path)
    X_v3 = pipeline.transform(df[SENSOR_COLS])
    cluster_ids = X_v3['cluster_id'].values if 'cluster_id' in X_v3.columns else None
    print(f'V3 features: {X_v3.shape[1]}')
else:
    cluster_ids = None; print('Pipeline not found. Run p3_lstm_hybrid_rul.ipynb first.')
X_scaled = StandardScaler().fit_transform(X_raw)
print(f'Dataset: {df.shape}')

## PCA — Linear Dimensionality Reduction

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print(f'Explained variance: {pca.explained_variance_ratio_.round(3)}, total={pca.explained_variance_ratio_.sum()*100:.1f}%')
fig, axes = plt.subplots(1,3,figsize=(18,5))
sc = axes[0].scatter(X_pca[:,0],X_pca[:,1],c=y_failure,cmap='RdYlGn_r',alpha=0.3,s=5)
plt.colorbar(sc,ax=axes[0],label='Machine Failure'); axes[0].set_title('PCA — Failure')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
if cluster_ids is not None:
    sc2 = axes[1].scatter(X_pca[:,0],X_pca[:,1],c=cluster_ids,cmap='tab10',alpha=0.3,s=5)
    plt.colorbar(sc2,ax=axes[1],label='Cluster'); axes[1].set_title('PCA — Clusters')
else:
    axes[1].text(0.5,0.5,'Clusters not available',ha='center',va='center',transform=axes[1].transAxes)
loadings = pd.DataFrame(pca.components_.T,columns=['PC1','PC2'],
                        index=['AirTemp','ProcTemp','RPM','Torque','Wear'])
loadings.plot(kind='bar',ax=axes[2],color=['steelblue','coral'])
axes[2].set_title('PCA Loadings'); axes[2].axhline(0,color='k',linewidth=0.8)
plt.tight_layout(); plt.show()

## UMAP — Nonlinear Dimensionality Reduction

In [ ]:
if HAS_UMAP:
    print('Running UMAP (~30s)...')
    reducer = umap.UMAP(n_components=2,n_neighbors=15,min_dist=0.1,random_state=42)
    X_umap = reducer.fit_transform(X_scaled)
    fig,axes = plt.subplots(1,2,figsize=(16,6))
    sc = axes[0].scatter(X_umap[:,0],X_umap[:,1],c=y_failure,cmap='RdYlGn_r',alpha=0.4,s=5)
    plt.colorbar(sc,ax=axes[0],label='Failure'); axes[0].set_title('UMAP — Failure')
    if cluster_ids is not None:
        sc2 = axes[1].scatter(X_umap[:,0],X_umap[:,1],c=cluster_ids,cmap='tab10',alpha=0.4,s=5)
        plt.colorbar(sc2,ax=axes[1],label='Cluster'); axes[1].set_title('UMAP — Clusters')
    else:
        axes[1].text(0.5,0.5,'Clusters not available',ha='center',va='center',transform=axes[1].transAxes)
    plt.tight_layout(); plt.show()
    dist = float(np.linalg.norm(X_umap[y_failure==1].mean(0)-X_umap[y_failure==0].mean(0)))
    print(f'Centroid distance failures vs healthy: {dist:.3f} (higher = more separable)')
else:
    print('UMAP not available. pip install umap-learn')

## Anomaly Score Overlay on UMAP

In [ ]:
if not HAS_UMAP: print('Requires UMAP.'); raise SystemExit
p4v2 = os.path.join(MODELS_DIR,'ml_model_p4_anomaly_v2.pkl')
p4v1 = os.path.join(MODELS_DIR,'ml_model_p4_anomaly.pkl')
if os.path.exists(p4v2):
    d = joblib.load(p4v2); iso = d['iso_model']; label='P4 V2 (IF component)'
elif os.path.exists(p4v1):
    d = joblib.load(p4v1)
    iso = d.get('model',d) if isinstance(d,dict) else d; label='P4 V1 IF'
else:
    print('No P4 model found.'); raise SystemExit
scores = -iso.decision_function(X_raw)
fig,ax = plt.subplots(figsize=(10,7))
sc = ax.scatter(X_umap[:,0],X_umap[:,1],c=scores,cmap='YlOrRd',alpha=0.5,s=5)
plt.colorbar(sc,ax=ax,label='Anomaly Score')
ax.set_title(f'UMAP — {label}'); plt.tight_layout(); plt.show()
print('Anomalies (high score) should be at periphery.')